# Automatisation de la cartographie des eaux de surface grâce à l'IA


Ce notebook a été développé pour le webinaire premium de l'Australian Water School sur l'automatisation de la cartographie des eaux de surface grâce à l'IA, qui aura lieu le 23 juillet 2025. Pour vous inscrire, veuillez consulter le site <https://awschool.com.au/training/ai-tools-for-mapping>.

Ressources supplémentaires :

- **GitHub** : <https://github.com/opengeos/geoai>

- **Documentation** : <https://opengeoai.org>

- **Carnet de notes** : <https://opengeoai.org/workshops/GeoAI_Workshop_2025>

- **Application web** : <https://huggingface.co/spaces/giswqs/surface-water-app>

- **Démo de l’application web** : <https://youtu.be/LKySb6OYU7M>

- **Tutoriels YouTube** : <https://tinyurl.com/GeoAI-Tutorials>

## Utilisation du GPU Colab

Pour utiliser le GPU, cliquez sur le menu « Exécution » et sélectionnez « Modifier le type d’exécution ». Sélectionnez ensuite « T4 GPU » dans le menu déroulant.

## Installation des packages

Décommentez la cellule suivante pour installer le package. L’installation peut prendre quelques minutes. Veuillez patienter.

In [ ]:
# %pip install geoai-py

## Importer les bibliothèques

Ensuite, nous devons installer le paquet `geoai`.

In [ ]:
import geoai

## Cartographie des eaux de surface à partir d'images satellites non géoréférencées

Dans la première partie de ce notebook, nous allons montrer comment cartographier les eaux de surface à l'aide d'images satellites non géoréférencées au format JPG/PNG.

## Téléchargement des données d'exemple

Nous utiliserons le jeu de données [waterbody dataset](https://www.kaggle.com/datasets/franciscoescobar/satellite-images-of-water-bodies) disponible sur Kaggle. Vous devrez créer un compte et télécharger le jeu de données. Je l'ai déjà téléchargé et j'en ai enregistré une copie sur Hugging Face. Téléchargeons le jeu de données :

In [ ]:
url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/waterbody-dataset.zip"

In [ ]:
out_folder = geoai.download_file(url)

Le jeu de données décompressé contient deux dossiers : `images` et `masks`. Chaque dossier contient 2 841 images au format JPG. Le dossier `images` contient les images satellites originales, et le dossier `masks` contient les masques d'eau de surface correspondants. Veuillez noter que la taille des images varie considérablement.

Nous utiliserons ces 2 841 paires d'images pour entraîner un modèle de cartographie des eaux de surface.

### Entraînement d'un modèle de segmentation sémantique

Avant de nous plonger dans l'entraînement, comprenons les concepts fondamentaux des architectures et des encodeurs d'apprentissage profond.

L'architecture d'apprentissage profond est comme le **plan** d'un réseau de neurones : elle définit la structure du réseau et le flux des données. Elle comprend des couches de nœuds (neurones) qui traitent les données étape par étape pour apprendre des modèles, comme la reconnaissance de chats sur des photos ou la traduction de langues.

Il existe de nombreux types d'architectures :

* **Réseaux de neurones à propagation avant** (simples, à sens unique)

* **Réseaux de neurones convolutifs (CNN)** (utilisés pour les images)

* **Réseaux de neurones récurrents (RNN)** (utilisés pour les séquences comme la parole)

* **Transformateurs** (utilisés pour les tâches de traitement du langage, comme ChatGPT)

Chacune est conçue pour des types de problèmes spécifiques.

Un **encodeur** est une partie du réseau neuronal qui prend en entrée une donnée (comme une phrase ou une image) et la compresse en une forme plus compacte et exploitable appelée **représentation de caractéristiques** ou **plongement**. Il extrait les informations importantes tout en éliminant le bruit.

Par exemple, si vous soumettez la phrase « J'adore la pizza » à l'encodeur, celui-ci la transforme en une série de nombres qui conservent sa signification, mais sont plus faciles à comprendre et à traiter pour l'ordinateur.

Les encodeurs sont utilisés dans des modèles tels que :

* **Autoencodeurs** (pour la compression et la reconstruction de données)

* **Encodeurs de type transformateur** (comme BERT, pour la compréhension du langage)

* **Modèles encodeur-décodeur** (comme les systèmes de traduction automatique)

Nous allons maintenant entraîner un modèle de segmentation sémantique à l'aide de la nouvelle fonction `train_segmentation_model`. Cette fonction prend en charge différentes architectures du package `segmentation-models-pytorch` :

- **Architectures** : `unet`, `unetplusplus`, `deeplabv3`, `deeplabv3plus`, `fpn`, `pspnet`, `linknet`, `manet`
- **Encodeurs** : `resnet34`, `resnet50`, `efficientnet-b0`, `mobilenet_v2`, etc.

Pour plus de détails, veuillez consulter la [documentation de `segmentation-models-pytorch`](https://smp.readthedocs.io/en/latest/models.html).

Entraînons le module avec U-Net et l'encodeur ResNet34 :

In [ ]:
# Tester le modèle train_segmentation_model avec détection automatique de la taille
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/masks",
    output_dir=f"{out_folder}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=3, # Nombre de canaux de l'image d'entrée
    num_classes=2, # Fond et eau
    batch_size=32, # Nombre d'images à traiter par lot
    num_epochs=3, # Entraînement sur 3 époques pour gagner du temps ; en pratique, il est recommandé d'entraîner sur plus d'époques
    learning_rate=0.001, # Taux d'apprentissage de l'optimiseur
    val_split=0.2, # 20 % des données pour la validation
    target_size=(512, 512), # Taille cible de l'image d'entrée
    verbose=True, # Afficher la progression
)

Dans le dossier de sortie du modèle `unet_models`, vous trouverez les fichiers suivants :

- `best_model.pth` : Meilleur point de contrôle du modèle
- `final_model.pth` : Dernier point de contrôle du modèle
- `training_history.pth` : Historique de l'entraînement
- `training_summary.txt` : Résumé de l'entraînement

### Évaluation du modèle

Examinons les courbes d'apprentissage et les performances du modèle :

In [ ]:
geoai.plot_performance_metrics(
    history_path=f"{out_folder}/unet_models/training_history.pth",
    figsize=(15, 5),
    verbose=True,
)

![image](https://github.com/user-attachments/assets/381ce436-3520-4706-9def-b0a7ae8244ac)

### Effectuer une inférence sur une seule image

Vous pouvez effectuer une inférence sur une nouvelle image à l'aide de la fonction `semantic_segmentation`. Je ne dispose pas d'une nouvelle image pour effectuer le test ; j'utiliserai donc une des images d'entraînement. En pratique, vous utiliseriez vos propres images, non utilisées lors de l'entraînement.

In [ ]:
index = 3
test_image_path = f"{out_folder}/images/water_body_{index}.jpg"
ground_truth_path = f"{out_folder}/masks/water_body_{index}.jpg"
prediction_path = f"{out_folder}/prediction/water_body_{index}.png"  # save as png to preserve exact values and avoid compression artifacts
model_path = f"{out_folder}/unet_models/best_model.pth"

In [ ]:
geoai.semantic_segmentation(
    input_path=test_image_path,
    output_path=prediction_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=32,
)

In [ ]:
fig = geoai.plot_prediction_comparison(
    original_image=test_image_path,
    prediction_image=prediction_path,
    ground_truth_image=ground_truth_path,
    titles=["Original", "Prediction", "Ground Truth"],
    figsize=(15, 5),
    save_path=f"{out_folder}/prediction/water_body_{index}_comparison.png",
    show_plot=True,
)

![image](https://github.com/user-attachments/assets/00308228-0819-4161-9a35-6a98f4cefa93)

### Effectuer une inférence sur plusieurs images

Commençons par télécharger les images et les masques de test.

In [ ]:
url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/waterbody-dataset-sample.zip"

In [ ]:
data_dir = geoai.download_file(url)
images_dir = f"{data_dir}/images"
masks_dir = f"{data_dir}/masks"
predictions_dir = f"{data_dir}/predictions"

In [ ]:
geoai.semantic_segmentation_batch(
    input_dir=images_dir,
    output_dir=predictions_dir,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=4,
    quiet=True,
)

## Cartographie des eaux de surface avec l'imagerie Sentinel-2

Dans la deuxième partie de ce document, nous allons montrer comment cartographier les eaux de surface à l'aide d'images Sentinel-2 comportant six bandes spectrales : bleu, vert, rouge, proche infrarouge et infrarouge à ondes courtes.

## Téléchargement des données d'exemple

Nous utiliserons le jeu de données [Earth Surface Water Dataset](https://zenodo.org/records/5205674#.Y4iEFezP1hE) de Zenodo. Crédits à l'auteur (Xin Luo) du jeu de données.

In [ ]:
url = "https://zenodo.org/records/5205674/files/dset-s2.zip?download=1"
data_dir = geoai.download_file(url)

Dans le jeu de données décompressé, nous avons quatre dossiers :

- `dset-s2/tra_scene` : images d'entraînement
- `dset-s2/tra_truth` : masques d'entraînement
- `dset-s2/val_scene` : images de validation
- `dset-s2/val_truth` : masques de validation

Nous utiliserons les images et les masques d'entraînement pour entraîner un modèle de segmentation sémantique.

In [ ]:
images_dir = f"{data_dir}/dset-s2/tra_scene"
masks_dir = f"{data_dir}/dset-s2/tra_truth"
tiles_dir = f"{data_dir}/dset-s2/tiles"

### Création des données d'entraînement

Nous allons créer les mêmes tuiles d'entraînement qu'auparavant.

In [ ]:
result = geoai.export_geotiff_tiles_batch(
    images_folder=images_dir,
    masks_folder=masks_dir,
    output_folder=tiles_dir,
    tile_size=512,
    stride=128,
    quiet=True,
)

### Entraînement du modèle de segmentation sémantique

Nous allons maintenant entraîner un modèle de segmentation sémantique à l'aide de la nouvelle fonction `train_segmentation_model`. Entraînons le module avec U-Net et l'encodeur ResNet34 :

In [ ]:
geoai.train_segmentation_model(
    images_dir=f"{tiles_dir}/images",
    labels_dir=f"{tiles_dir}/masks",
    output_dir=f"{tiles_dir}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=6,
    num_classes=2,  # background and water
    batch_size=32,
    num_epochs=5,  # training for 5 epochs to save time, in practice you should train for more epochs
    learning_rate=0.001,
    val_split=0.2,
    verbose=True,
)

### Évaluation du modèle

Examinons les courbes d'apprentissage et les performances du modèle :

In [ ]:
geoai.plot_performance_metrics(
    history_path=f"{tiles_dir}/unet_models/training_history.pth",
    figsize=(15, 5),
    verbose=True,
)

![image](https://github.com/user-attachments/assets/61f675a7-ee67-4650-81c0-f754fe681f4d)

### Exécuter l'inférence

In [ ]:
images_dir = f"{data_dir}/dset-s2/val_scene"
masks_dir = f"{data_dir}/dset-s2/val_truth"
predictions_dir = f"{data_dir}/dset-s2/predictions"
model_path = f"{tiles_dir}/unet_models/best_model.pth"

In [ ]:
geoai.semantic_segmentation_batch(
    input_dir=images_dir,
    output_dir=predictions_dir,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=6,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=32,
    quiet=True,
)

### Visualiser les résultats

In [ ]:
test_image_path = (
    f"{data_dir}/dset-s2/val_scene/S2A_L2A_20190318_N0211_R061_6Bands_S2.tif"
)
ground_truth_path = (
    f"{data_dir}/dset-s2/val_truth/S2A_L2A_20190318_N0211_R061_S2_Truth.tif"
)
prediction_path = (
    f"{data_dir}/dset-s2/predictions/S2A_L2A_20190318_N0211_R061_6Bands_S2_mask.tif"
)
save_path = f"{data_dir}/dset-s2/S2A_L2A_20190318_N0211_R061_6Bands_S2_comparison.png"

fig = geoai.plot_prediction_comparison(
    original_image=test_image_path,
    prediction_image=prediction_path,
    ground_truth_image=ground_truth_path,
    titles=["Original", "Prediction", "Ground Truth"],
    figsize=(15, 5),
    save_path=save_path,
    show_plot=True,
    indexes=[5, 4, 3],
    divider=5000,
)

![image](https://github.com/user-attachments/assets/53601ed7-2bd6-4e7e-b369-4d7bfc2ce120)

### Télécharger les images Sentinel-2

In [ ]:
m = geoai.LeafMap(center=[-16.3043, 128.7412], zoom=10)
m.add_basemap("Esri.WorldImagery")
m.add_stac_gui()
m

In [ ]:
# m.stac_gdf

In [ ]:
# m.stac_item

In [ ]:
import leafmap

In [ ]:
url = "https://earth-search.aws.element84.com/v1/"
collection = "sentinel-2-l2a"
time_range = "2025-01-01/2025-07-20"
bbox = [128.6735, -16.2466, 128.9577, -16.0962]

In [ ]:
search = leafmap.stac_search(
    url=url,
    max_items=10,
    collections=[collection],
    bbox=bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 10}},
    sortby=[{"field": "properties.eo:cloud_cover", "direction": "asc"}],
    get_collection=True,
)
search

In [ ]:
search = leafmap.stac_search(
    url=url,
    max_items=10,
    collections=[collection],
    bbox=bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 10}},
    sortby=[{"field": "properties.eo:cloud_cover", "direction": "asc"}],
    get_gdf=True,
)
search.head()

In [ ]:
search = leafmap.stac_search(
    url=url,
    max_items=1,
    collections=[collection],
    bbox=bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 10}},
    sortby=[{"field": "properties.eo:cloud_cover", "direction": "asc"}],
    get_assets=True,
)
search

In [ ]:
bands = ["blue", "green", "red", "nir", "swir16", "swir22"]
assets = list(search.values())[0]
links = [assets[band] for band in bands]
for link in links:
    print(link)

In [ ]:
out_dir = "s2"
leafmap.download_files(links, out_dir)

### Empiler les bandes d'images

Décommentez la cellule suivante pour installer GDAL sur Colab.

In [ ]:
# !apt-get install -y gdal-bin

In [ ]:
s2_path = "s2.tif"
geoai.stack_bands(input_files=out_dir, output_file=s2_path)

In [ ]:
geoai.view_raster(s2_path, indexes=[4, 3, 2])

### Run inference on a Sentinel-2 image

In [ ]:
s2_mask = "s2_mask.tif"
model_path = f"{tiles_dir}/unet_models/best_model.pth"

In [ ]:
geoai.semantic_segmentation(
    input_path=s2_path,
    output_path=s2_mask,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=6,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=32,
)

### Visualize the results

In [ ]:
geoai.view_raster(
    s2_mask, no_data=0, colormap="viridis", basemap=s2_path, backend="ipyleaflet"
)


## Surface water mapping with aerial imagery

In the last part of this notebook, we will demonstrate how to map surface water using aerial imagery from the USDA National Agriculture Imagery Program (NAIP).

### Download sample data

In [ ]:
train_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_train.tif"
train_masks_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_masks.tif"
test_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_test.tif"

In [ ]:
train_raster_path = geoai.download_file(train_raster_url)
train_masks_path = geoai.download_file(train_masks_url)
test_raster_path = geoai.download_file(test_raster_url)

In [ ]:
geoai.print_raster_info(train_raster_path, show_preview=False)

### Visualize sample data

In [ ]:
geoai.view_raster(train_masks_url, nodata=0, basemap=train_raster_url)

In [ ]:
geoai.view_raster(test_raster_url)

### Create training data

In [ ]:
out_folder = "naip"

In [ ]:
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_masks_path,
    tile_size=512,
    stride=128,
    buffer_radius=0,
)

### Train segmentation model

In [ ]:
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=4,
    pretrained=True,
    batch_size=8,
    num_epochs=5,
    learning_rate=0.005,
    val_split=0.2,
)

### Evaluate the model

In [ ]:
geoai.plot_performance_metrics(
    history_path=f"{out_folder}/models/training_history.pth",
    figsize=(15, 5),
    verbose=True,
)

### Run inference

In [ ]:
masks_path = "naip_water_prediction.tif"
model_path = f"{out_folder}/models/best_model.pth"

In [ ]:
geoai.semantic_segmentation(
    test_raster_path,
    masks_path,
    model_path,
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    window_size=512,
    overlap=128,
    confidence_threshold=0.3,
    batch_size=32,
    num_channels=4,
)

### Vectorize masks

In [ ]:
output_path = "naip_water_prediction.geojson"
gdf = geoai.raster_to_vector(
    masks_path, output_path, min_area=1000, simplify_tolerance=1
)

In [ ]:
gdf = geoai.add_geometric_properties(gdf)

In [ ]:
len(gdf)

In [ ]:
geoai.view_vector_interactive(gdf, tiles=test_raster_url)

In [ ]:
gdf["elongation"].hist()

In [ ]:
gdf_filtered = gdf[gdf["elongation"] < 10]

In [ ]:
len(gdf_filtered)

### Visualize results

In [ ]:
geoai.view_vector_interactive(gdf_filtered, tiles=test_raster_url)

In [ ]:
geoai.create_split_map(
    left_layer=gdf_filtered,
    right_layer=test_raster_url,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_url,
)

![image](https://github.com/user-attachments/assets/a269b5a0-9f72-4ed8-8b2d-a175bbc45a23)